<h1 id='1' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">📋 Summary</h1>

<div>
<img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExemVreTF0Nm9vZWNxNjE3Z2RhMGo2dzFxZ2VhNDNpbnA2ZWFoajZ2dCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/xT9IgxJXKgMD96peHC/giphy.webp" width="600"/>
</div>

**About Dataset**

This dataset contains comprehensive information on 2,392 high school students, detailing their demographics, study habits, parental involvement, extracurricular activities, and academic performance. The target variable, GradeClass, classifies students' grades into distinct categories, providing a robust dataset for educational research, predictive modeling, and statistical analysis. This dataset offers a comprehensive view of the factors influencing students' academic performance, making it ideal for educational research, development of predictive models, and statistical analysis.

**Process**

My process in analyzing this dataset is to:
* Determine which features are most important for predicting academic success of students (defined here as Grade Class)
* Perform dimensionality reduction as necessary if certain features are not correlated to Grade Class
* Encode categorical variables and scale numerical variables
* Split the dataset into predictors (X) and target (y)
* Split the X and y datasets into training and testing sets
* Determine the best model for the dataset, fit it to the training set, and use this model to make predictions on the test set
* Tune the hyperparameters of the model to (hopefully) increase accuracy and address any over/underfitting

These steps will create a machine learning model capable of using the features in this dataset to predict the grade that a hypothetical student might receive.

**✅If this notebook helps you out, please upvote✅**

<h1 id='2' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">📚 Import Libraries</h1>

<div>
<img src="https://media0.giphy.com/media/v1.Y2lkPTc5MGI3NjExaGw3NmRhNGNpNHc2N3pzZGNkdHhxOTkwYjljNXl5YnF6bTkzbWNvdiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/fsXOS3oBboiYf6fSsY/giphy.webp" width="600"/>
</div>

In [ ]:
# Import useful Libraries
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV, GridSearchCV, train_test_split
from scipy.stats import loguniform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool, cv

# Silence Warnings (optional)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

<h1 id='3' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">📊 Preprocessing Data</h1>

<div>
<img src = 'https://media3.giphy.com/media/v1.Y2lkPTc5MGI3NjExZzMxNHA5endkdHI4eXhocmF1aDFqbmt4cHdyeDVhYWE3bGp5eXB4aiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/k1aqHacjJxN1xmFtyR/giphy.webp' width=600>
</div>

## Import dataset

The first step in analysis is to import the data (in this case a csv file) and read it into a pandas DataFrame so we can make transformations easily and work with a more efficient data structure. Pandas can also deal with any missing values by removing or imputing.

We start by creating a variable for the file path and combining the directory and filename.

In [ ]:
# Read filename from kaggle filepath
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Then we assign this path (join dirname + filename) to the variable df.

In [ ]:
# Create DataFrame from the CSV file
df = pd.read_csv(os.path.join(dirname, filename))

Let's take a quick look at the last 5 rows of the DataFrame we've created:

In [ ]:
df.tail()

Let's do a quick check for NAs as well:

In [ ]:
df.info()

We confirm from the info that there are no nulls in this DataFrame.

## Feature Distribution

We can look into the distribution of each feature and gain some insight into this dataset and how the features might be treated.

In [ ]:
df.hist(figsize=(20,10),bins=7, color='lightblue')

Looking at the histograms above I can make some quick inferences:
* StudentID has no distribution and logically would have no effect on Grade
* There are only 4 ages in this dataset, which strangely makes age a categorical feature
* There are only 4 ethnicity variables in this dataset
* There are a lot of low-scoring students in this dataset (a majority of 4s - Fs in GradeClass)

## Define Categorical and Numerical Features

In order to proceed with assessing feature importance, there are two important steps:
1. Determine which columns are numeric and which are categoric
2. Encode the categoric columns to turn object variables into numbers
3. Scale numerical columns to ensure that large numbers have an equal effect on our model as small numbers

In [ ]:
# Distinction is based on the number of different values in the column
columns = list(df.columns)

categoric_columns = []
numeric_columns = []

for i in columns:
    if len(df[i].unique()) > 5:
        numeric_columns.append(i)
    else:
        categoric_columns.append(i)
        
# Assuming the first column is an ID or non-numeric feature
numeric_columns = numeric_columns[1:]

print('Numerical features: ', numeric_columns)
print('Categorical features: ', categoric_columns)

I create an empty list for categoric columns and numeric columns, and then createa  for loop that cycles through our features and checks the number of unique values. If there are more than 5 unique values, we can assume that the feature is numerical. If there are less than 5, we can assume it's categorical. 

This isn't true in all instances, and we can see one thing here that is a little wierd: *Age is considered to be a categorical feature in this dataset.* 

The reason is that this dataset only contains high school students, so the age range is very small. If we were analyzing a dataset of students of all ages, then age would almost certainly be a numerical variable.

To ensure our numeric columns only contain numbers, I also convert the numeric columns to float64 type.

In [ ]:
# Convert numeric columns to float64
df[numeric_columns] = df[numeric_columns].astype('float64')

## Encode Categorical Features and Scale Numerical Features

There are different ways to encode features, for example One Hot Encoding. In this example I'm using LabelEncoder to encode the categoric features. I'm also using StandardScaler to scale our numeric columns to make them have equal weight on the model we use to evaluate feature importance and make predictions.

In [ ]:
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode categorical features
df = df.copy()
for column in df[categoric_columns]:  
    df[column] = label_encoder.fit_transform(df[column])

# Standardize numerical features
scaler = StandardScaler()
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])

## Correlation Among Features

Before I move onto splitting the dataset and assessing feature importance, I want to look closely at the correlation among features in this dataset.

In [ ]:
plt.figure(figsize=(16, 8))
sns.heatmap(df.corr(), annot = True, cmap = "coolwarm")
plt.title('The correlation among features',y= 1.05)
plt.show()

We can observe that the most correlated features to GradeClass are GPA and Absences. It is important to note that GPA cannot be treated as a predictor in this dataset. More information about this can be found below. When I split the DataFrame I will drop GPA.

## Split the Dataset

I'm going to split this dataset 2 ways before proceeding:
1. Split the data into our predictors (X) and our target feature (y)
2. Split the X and y dataframes into training and testing sets

Note: I'm also dropping some features that I don't want to add noise to the model here:
* GradeClass <- our target feature
* StudentID <- studentID is not logically a useful predictor of a student's grade
* Age <- I thought about this for a while, and I do not believe age should be used as a predictor for this data. The feature would not generalize well to future predictions on unseen data unless the age range is the same (15-18). I believe that removing age as a predictor creates a more robust model that we can apply to future student performance analysis in any academic setting (i.e. elementary school or university).
* GPA <- this would add unnecessary noise to the model and is not logical for future predictions

**For more information on why you should not use GPA as a predictor please see: [🚫Why You Shouldn't Use GPA to Predict Class Grade❌](https://www.kaggle.com/datasets/rabieelkharoua/students-performance-dataset/discussion/512561)**

In [ ]:
# CHOOSE THE TARGET FEATURE HERE, IN THIS CASE IT IS 'GradeClass'
X = df.drop(columns=['GradeClass', 'GPA', 'StudentID', 'Age'])
y = df['GradeClass']

# Splitting the data into training and testing sets (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Feature Importance

I always find this step really cool, we can use a classification model to determine the importance of features in our dataset in terms of predicting our target variable. In this case, I'm using a RandomForestClassifier to assess the relative importance of features to predicting y.

In [ ]:
clf = RandomForestClassifier(random_state = 42)
clf = clf.fit(X, y)

fimp = pd.Series(data=clf.feature_importances_, index=X.columns).sort_values(ascending=False)

Let's plot the results (a pandas series) in a barplot to make it obvious:

In [ ]:
plt.figure(figsize=(17,13))
plt.title("Feature importance")
ax = sns.barplot(y=fimp.index, x=fimp.values, orient='h')

The most important predictor of grade - by a long shot - is absences. This makes sense, but is also something I might consider removing later on. I am curious if the model will be equally (or more) accurate if we remove absence as a predictor. Of course, if a student is absent for many classes, they are likely to receive a lower grade. Does this actually reflect the student's academic abilities though?

For example, consider a student who is sick for a long period of time, or dealing with issues beyond their control at home. Yes, they will receive a lower grade, but their potential is certainly higher than the model would reflect. This is food for thought in regards to this particular dataset and question.

<h1 id='4' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">🏗️ Building a Model</h1>

<div>
<img src = 'https://blogger.googleusercontent.com/img/b/R29vZ2xl/AVvXsEidnEOgFwpnPCWc2MS0Zpo8WdcPj-8M8rZ6hX3THlZ-7X4s2nkH579fMzY0V-XI2hrxX6X-qHnL3SxIv01DTromECkv0ZPh7CIZy2bbheFRTdhF54YwWkrp74keRwfgN5TO00qz-5YwYHDB/s640-rw/gunpla+builders+%25281%2529.gif' width=600>
</div>

## Selecting a Classification Model

I really like this idea of cycling through all models with default settings and using the one with the best score as a starting block. It is *not* 100% accurate though. There is always a chance that after tuning a different model could yield better results. This is especially true in situations where multiple models receive similar scores. 

In this case, I create a dictionary of classification models and 2 empty lists - model_names and accuracies. I use a for loop to loop through the dictionary and fit the models one-by-one to the training data. I then use the fitted model to make a prediction on the y_test data and score it using clf.score function. I append model names to the model_names list and scores to the accuracies list respectively. Finally, I create a dataframe with the model_names and accuracies and plot it using a barplot to easily visualize the most accurate models.

In [ ]:
# Dictionary of classification models
classification_models = {
    "Logistic Regression": LogisticRegression(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
    "XGBoost": XGBClassifier(),
    "CatBoost": CatBoostClassifier(silent=True),
}

model_names = []
accuracies = []

# Train and evaluate each model
for name, clf in classification_models.items():
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    model_names.append(name)
    accuracies.append(score)
    print(f"{name} accuracy: {score:.2f}")

# Create a DataFrame for model accuracies
df_models = pd.DataFrame({'Model': model_names, 'Accuracy': accuracies})

# Plot model accuracies using Plotly
fig = px.bar(df_models, x='Model', y='Accuracy', title='Model Accuracies')
fig.show()

We can see that there are multiple models that would be effective for this task:
* Logistic Regression
* Support Vector Machine
* Random Forest
* Gradient Boosting
* XGBoost
* Catboost

In [ ]:
# Find the best model
best_index = accuracies.index(max(accuracies))
best_model_name = model_names[best_index]
best_model = classification_models[best_model_name]

print(f"The best model is: {best_model_name} with an accuracy of {accuracies[best_index]:.2f}")


I can easily select the best model from our dictionary by sorting by accuracy. In this case, the model I will start with is a Support Vector Machine.

<h1 id='5' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">✔️ Evaluating the Model</h1>

<div>
    <img src ='https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExbnJiYnJnMzFla2Ywa245ZnZpeDZ6eGJlYmNlMWNpZGc3eXF6bGZucSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9dg/6oC7Q5NBUPZ49HnV3S/giphy.gif' width=600>
    </div>

## Model Evaluation

I fit the best model to the training data, and use the fitted model to make predictions for y_test. We then write these scores into a Confusion Matrix to quickly evaluate how the model performed. We can see that the model is approximately 74.5% accurate in it's predictions. There are a lot of misclassified predictions. 74.5% is pretty good, but not perfect. 

In [ ]:
# Initialize and train model
best_model.fit(X_train, y_train)
model_score = best_model.score(X_test, y_test)
y_pred = best_model.predict(X_test)

# Calculate and plot the confusion matrix
score = round(accuracy_score(y_test, y_pred), 3)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=".0f")
plt.xlabel('Predicted Values')
plt.ylabel('Actual Values')
plt.title('Accuracy Score: {0}'.format(score), size=15)
plt.show()

We can also compare the results for predicting the test data to the predictions on training data to check for over/underfitting.

<h1 id='6' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">🚀 Tuning the Model</h1>

<div>
    <img src='https://media1.giphy.com/media/v1.Y2lkPTc5MGI3NjExcmdkbjVyM3FkcWU4Yzk5dDJpbm85NXdpZXhuczV3cmk5c3h4cm9rNyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/xT5LMxPzeCh2wAIGqI/giphy.webp' width=600>
    </div>

## Hyperparameter Tuning

Hyperparameter tuning can be quite complicated. A simple approach is to use a grid or randomized search to determine the best hyperparameters. A grid search is generally more powerful, at the cost of increased resources. A randomized search might not be 100% optimal, but it is generally faster. 

In this case I've selected a RandomizedSearch due to the amount of resources on my PC. In order to implement a RandomizedSearch I define the evaluation metrics (RepeatedStratifiedKFold with 10 splits) and the model (Support Vector Machine) and then create a grid of potential hyperparameters before using Randomized Search to make a guess at the optimal configurations from random combinations from the grid.

In [ ]:
# Define the model
model = SVC()

# Define evaluation
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

# Define grid
grid = {'C': [0.1, 1, 10, 100, 1000],  
        'gamma': [1, 0.1, 0.01, 0.001, 0.0001], 
        'kernel': ['rbf']
       }

# Define search
search = GridSearchCV(estimator=model, param_grid=grid, cv=cv, scoring='accuracy', n_jobs=-1)

# Fit search to training data
result = search.fit(X_train, y_train)

# Summarize result
print('Best Score: %s' % result.best_score_)
print('Best Hyperparameters: %s' % result.best_params_)

## Model Evaluation (again)

In [ ]:
# Initialize and train model
score = result.score(X_test, y_test)
y_pred = result.predict(X_test)

# Calculate and plot the confusion matrix
score = round(accuracy_score(y_test, y_pred), 3)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=".0f")
plt.xlabel('Predicted Values')
plt.ylabel('Actual Values')
plt.title('Accuracy Score: {0}'.format(score), size=15)
plt.show()

If we use the optimized parameters selected through hyperparameter tuning, we find that our model is slightly less accurate when predicting the test data. I find this curious, I would have thought that adjusting hyperparameters through GridSearch might increase the accuracy of our model. 

## Reducing Dimensionality

We know from our earlier investigation that the most important features are:

In [ ]:
fimp.head(3)

What happens to our predictions if we only use these as predictors?

In [ ]:
X_train = X_train[['Absences', 'StudyTimeWeekly', 'ParentalSupport']]
X_test = X_test[['Absences', 'StudyTimeWeekly', 'ParentalSupport']]

In [ ]:
# Define Model
model = GradientBoostingClassifier(n_estimators = 50, max_depth = 5, learning_rate = 0.1)

# Fit search to new training data
result = model.fit(X_train, y_train)

# Initialize and train model
score = result.score(X_test, y_test)
y_pred = result.predict(X_test)

# Calculate and plot the confusion matrix
score = round(accuracy_score(y_test, y_pred), 3)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=".0f")
plt.xlabel('Predicted Values')
plt.ylabel('Actual Values')
plt.title('Accuracy Score: {0}'.format(score), size=15)
plt.show()

This is actually quite interesting. The accuracy of our model drops slightly, but the dimensionality is massively reduced. It turns out that we can predict the academic success of students with almost 70% accuracy using just 3 features.

<h1 id='7' style="padding:20px; 
            color:#000000;
            margin:10px;
            font-size:220%;
            text-align:center;
            display:fill;
            border-radius:20px;
            border-width: 5px;
            border-style: solid;
            border-color:#000000;
            background-color:#B6DE8D;
            overflow:hidden;
            font-weight:500">🏁 Conclusion</h1>

<div>
<img src = 'https://media3.giphy.com/media/v1.Y2lkPTc5MGI3NjExNHd0MHF6eHp5dWYycXB0YTB3dGhodXJveXZhaHM0aDY3YngxdnQ0OSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3oEjI8Kq5HhZLCrqBW/giphy.webp' width = 600>
</div>

A Support Vector Machine predicts the grade of students using the features provided with 74.5% accuracy on the test set without hyperparameter tuning.

The same model is 69.7% accurate if we reduce the dimensionality of the DataFrame to just 3 features:
1. Absences            
2. StudyTimeWeekly     
3. ParentalSupport    

If we want our model to be slightly more accurate at the cost of more resources we can include all of the features.     

There's a really interesting (and perhaps obvious) takeaway from this. If you want to achieve good grades in highschool:
* Go to class
* Study the material you learn in class


Parental support - or the involvement of parents in your academic endeavours - is an important predictor in the success of the student. Unfortunately for students, this feature is beyond their control.

**✅If this notebook helps you out, please upvote✅**

<div>
<img src = 'https://c.tenor.com/UMHdiHRwRAsAAAAC/tenor.gif')>
</div>